<a href="https://colab.research.google.com/github/ikbalktlk-crypto/rezervasyonsis/blob/main/Rezervasyonsis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q "pandas==2.2.2" "google-auth==2.47.0" "google-genai" openai

Kütüphane kurulumları

In [3]:

import os

def dosya_bul(isim):
    for kok, klasorler, dosyalar in os.walk('/'):
        if isim in dosyalar:
            return os.path.join(kok, isim)
    return None

gercek_yol = dosya_bul('otel_sistemi(Odalar).csv')
print(f"BULDUM! Dosyanın gizli adresi: {gercek_yol}")

BULDUM! Dosyanın gizli adresi: /otel_sistemi(Odalar).csv


In [6]:
import pandas as pd
import sqlite3


dosya_yolu = '/otel_sistemi(Odalar).csv'

try:

    df = pd.read_csv(dosya_yolu, sep=';', encoding='iso-8859-9')


    df.columns = [col.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')
                  .replace('ı', 'i').replace('ş', 's').replace('ç', 'c')
                  .replace('ö', 'o').replace('ü', 'u').replace('ğ', 'g')
                  .replace('.', '').replace('?', 'i').replace('-', '_') for col in df.columns]

    conn = sqlite3.connect('otel_yonetim.db')
    df.to_sql('Odalar', conn, if_exists='replace', index=False)


    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Rezervasyonlar (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            oda_id INTEGER,
            misafir_adi TEXT,
            giris DATE,
            cikis DATE,
            FOREIGN KEY (oda_id) REFERENCES Odalar(ID)
        )
    ''')
    conn.commit()
    conn.close()

    print("--- BAŞARILI ---")
    print(f"Toplam {len(df)} oda 'otel_yonetim.db' dosyasına başarıyla yüklendi.")
    print("Artık bu veritabanı üzerinden rezervasyon alabiliriz!")

except Exception as e:
    print(f"Bir hata oluştu: {e}")

--- BAŞARILI ---
Toplam 500 oda 'otel_yonetim.db' dosyasına başarıyla yüklendi.
Artık bu veritabanı üzerinden rezervasyon alabiliriz!


otel_sistemi(Odalar).csv verisini okuyup veritabanı kurulması.

In [7]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('otel_yonetim.db')


sorgu_suite = "SELECT Oda_No, Tip, Manzara, Fiyat_TL FROM Odalar WHERE Tip='Suite' LIMIT 2"
suite_odalar = pd.read_sql(sorgu_suite, conn)


sorgu_genel = "SELECT Oda_No, Kat, Bina_Blok, Klima, Mini_Buzdolabi FROM Odalar LIMIT 3"
genel_odalar = pd.read_sql(sorgu_genel, conn)

print("--- SUITE ODA TESTİ ---")
print(suite_odalar)
print("\n--- GENEL DONANIM TESTİ ---")
print(genel_odalar)

conn.close()

--- SUITE ODA TESTİ ---
  Oda_No    Tip Manzara  Fiyat_TL
0  A-103  Suite   Deniz      2500
1  A-106  Suite   Bahçe      2350

--- GENEL DONANIM TESTİ ---
  Oda_No  Kat Bina_Blok Klima Mini_Buzdolabi
0  A-101    1    A Blok   Var            Var
1  A-102    1    A Blok   Var            Var
2  A-103    1    A Blok   Var            Var


Oluşturduğum veritabanının doğruluğunu kontrol ettim.

In [14]:

import os
import sqlite3
import pandas as pd
import json
import smtplib
from email.message import EmailMessage
from openai import OpenAI
from google.colab import userdata
from datetime import datetime

os.environ["OPENAI_API_KEY"] = userdata.get('personal_key')
client = OpenAI()

GMAIL_ADRESIM = "ikbalkutluk2004@gmail.com"
GMAIL_UYGULAMA_SIFRESI = userdata.get('GMAIL_PASS')
bugun_tarih = datetime.now().strftime('%Y-%m-%d')

mesaj_gecmisi = [
    {
        "role": "system",
        "content": f"""Sen tam yetkili bir otel asistanısın. Bugünün tarihi: {bugun_tarih}.
        Rezervasyon yapma, oda sorgulama ve e-posta gönderme yetkilerine sahipsin.

        KURALLAR:
        1. {bugun_tarih} tarihinden öncesine kesinlikle rezervasyon yapılamaz.
        2. Müsait oda yoksa alternatif tarihler öner.
        3. Kullanıcıya kaç kişi olduklarını ve oda tipi tercihlerini (Single, Double, Suite) sor.
        4. KESİN KURAL: Oda özelliklerini (Manzara, Klima, Yangın Alarmı vb.) ASLA kendin uydurma. 'bos_oda_bul' fonksiyonundan gelen JSON verisindeki kelimeleri birebir kullan. Veride 'Deniz' yazıyorsa 'Deniz' yaz, veride yoksa o özellikten bahsetme.
        5. Oluşturduğumuz veritabanın içerdiği özelliklerin dışına çıkma.
        6. Veritabanındaki fiyatlar odaların gecelik fiyatıdır.Kalınacak gün,gece sayısına göre kullanıcıya toplam fiyat bilgisini ver.
        7. Rezervasyon oluşturulurken her odada yangın alarmının ve klimanın var olduğunu kullanıcıya aktar.
        8. Rezervasyon oluştururken veya oda önerirken veritabanındaki oda no bilgilerinden farklı bir oda no oluşturma.
        9. Belli bir tarih aralığında rezerve edilmiş odayı ASLA rezerve edildiği tarih aralığında başka bir kullanıcıya önerme.
        10. Kullanıcı aynı oda tipi, aynı manzaralı odaların fiyat farkının sebebini sorduğunda veritabanındaki değişkenlerden olan Balkon Tipi (Fransız balkon,Teras,Balkonlu,Balkonsuz) olduğuyla ilgili bir açıklama yap.
        11. Müşteriye verdiğin bilgileri mesaj_gecmisi üzerinden takip et.
        12. Müşterinin girdiği bilgileri unutma, hafızanda tut.
        13. Mail gönderirken 'mail_at' aracını kullan.
        14. Mail gönderirken statik cümleler yerine veritabanından bulduğun Oda_No, Tip, Manzara ve Fiyat_TL bilgilerini kullanarak detaylı bir özet geç.
        15. Mail gönderirken mesajın başında sayın misafirimiz gibi hitaplar ve mesajın sonunda iyi günler dilerim, daha fazla bilgi edinmek için müşteri hizmetlerimizle iletişime geçebilirisiniz gibi ifadeler kullan.
        16. Mail gönderirken madde gibi değil daha samimi bir dille mesaj yaz."""
    }
]

def bos_oda_bul(giris_tarihi, cikis_tarihi, oda_tipi):
    if giris_tarihi < bugun_tarih:
        return {"hata": f"Giriş tarihi ({giris_tarihi}) geçmiş bir tarihtir."}
    conn = sqlite3.connect('otel_yonetim.db')
    query = """
    SELECT ID, Oda_No, Tip, Manzara, Fiyat_TL FROM Odalar
    WHERE Tip = ? AND ID NOT IN (
        SELECT oda_id FROM Rezervasyonlar
        WHERE (giris < ? AND cikis > ?)
    ) LIMIT 3
    """
    df = pd.read_sql_query(query, conn, params=(oda_tipi, cikis_tarihi, giris_tarihi))
    conn.close()
    return df.to_dict(orient='records')

def rezervasyon_tamamla(misafir_adi, oda_id, giris, cikis):
    if giris < bugun_tarih:
        return f"Hata: {giris} tarihi bugünden öncedir."
    conn = sqlite3.connect('otel_yonetim.db')
    cursor = conn.cursor()
    check_query = "SELECT id FROM Rezervasyonlar WHERE oda_id = ? AND (giris < ? AND cikis > ?)"
    cursor.execute(check_query, (oda_id, cikis, giris))
    if cursor.fetchone():
        conn.close()
        return "Hata: Bu oda bu tarihlerde dolu."
    cursor.execute("INSERT INTO Rezervasyonlar (oda_id, misafir_adi, giris, cikis) VALUES (?,?,?,?)",
                   (oda_id, misafir_adi, giris, cikis))
    conn.commit()
    conn.close()
    return f"İşlem Başarılı! {misafir_adi} adına {oda_id} ID'li oda rezerve edildi."

def rezervasyon_sorgula(misafir_adi=None, oda_id=None):
    try:
        conn = sqlite3.connect('otel_yonetim.db')
        sorgu = "SELECT * FROM Rezervasyonlar WHERE 1=1"
        params = []
        if misafir_adi:
            sorgu += " AND misafir_adi LIKE ?"
            params.append(f"%{misafir_adi}%")
        if oda_id:
            sorgu += " AND oda_id = ?"
            params.append(oda_id)
        df = pd.read_sql_query(sorgu, conn, params=params)
        conn.close()
        return df.to_string(index=False) if not df.empty else "Uygun kayıt bulunamadı."
    except Exception as e:
        return f"Sorgulama hatası: {str(e)}"

def rezervasyon_iptal(oda_id, misafir_adi):
    conn = sqlite3.connect('otel_yonetim.db')
    cursor = conn.cursor()
    cursor.execute("DELETE FROM Rezervasyonlar WHERE oda_id = ? AND misafir_adi = ?", (oda_id, misafir_adi))
    conn.commit()
    rows = cursor.rowcount
    conn.close()
    return "İşlem Başarılı: Rezervasyon silindi." if rows > 0 else "Hata: Kayıt bulunamadı."

def guncel_rezervasyonlari_goster():
    try:
        conn = sqlite3.connect('otel_yonetim.db')
        df = pd.read_sql_query("SELECT * FROM Rezervasyonlar ORDER BY id DESC", conn)
        conn.close()
        return df.to_string(index=False) if not df.empty else "Henüz rezervasyon yok."
    except Exception as e:
        return f"Hata: {str(e)}"

def mail_at(alici_mail, konu, icerik):
    msg = EmailMessage()
    msg.set_content(icerik)
    msg['Subject'] = konu
    msg['To'] = alici_mail
    msg['From'] = GMAIL_ADRESIM
    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(GMAIL_ADRESIM, GMAIL_UYGULAMA_SIFRESI)
            smtp.send_message(msg)
        return "Mail başarıyla gönderildi!"
    except Exception as e:
        return f"Mail hatası: {str(e)}"

tools = [
    {"type": "function", "function": {"name": "bos_oda_bul", "description": "Müsait odaları listeler.", "parameters": {"type": "object", "properties": {"giris_tarihi": {"type": "string", "description": "YYYY-MM-DD"}, "cikis_tarihi": {"type": "string", "description": "YYYY-MM-DD"}, "oda_tipi": {"type": "string", "enum": ["Single", "Double", "Suite"]}}, "required": ["giris_tarihi", "cikis_tarihi", "oda_tipi"]}}},
    {"type": "function", "function": {"name": "rezervasyon_tamamla", "description": "Rezervasyon kaydı yapar.", "parameters": {"type": "object", "properties": {"misafir_adi": {"type": "string"}, "oda_id": {"type": "integer"}, "giris": {"type": "string"}, "cikis": {"type": "string"}}, "required": ["misafir_adi", "oda_id", "giris", "cikis"]}}},
    {"type": "function", "function": {"name": "rezervasyon_sorgula", "description": "Rezervasyon arar.", "parameters": {"type": "object", "properties": {"misafir_adi": {"type": "string"}, "oda_id": {"type": "integer"}}, "required": []}}},
    {"type": "function", "function": {"name": "rezervasyon_iptal", "description": "Rezervasyonu siler.", "parameters": {"type": "object", "properties": {"oda_id": {"type": "integer"}, "misafir_adi": {"type": "string"}}, "required": ["oda_id", "misafir_adi"]}}},
    {"type": "function", "function": {"name": "guncel_rezervasyonlari_goster", "description": "Tüm tabloyu gösterir.", "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {"name": "mail_at", "description": "Onay maili gönderir.", "parameters": {"type": "object", "properties": {"alici_mail": {"type": "string"}, "konu": {"type": "string"}, "icerik": {"type": "string"}}, "required": ["alici_mail", "konu", "icerik"]}}}
]

def sohbet_et(mesaj):
    mesaj_gecmisi.append({"role": "user", "content": mesaj})

    response = client.chat.completions.create(model="gpt-4o-mini", messages=mesaj_gecmisi, tools=tools)
    response_message = response.choices[0].message
    mesaj_gecmisi.append(response_message)

    if response_message.tool_calls:
        for tool_call in response_message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            func_name = tool_call.function.name

            func_map = {
                "bos_oda_bul": bos_oda_bul,
                "rezervasyon_tamamla": rezervasyon_tamamla,
                "rezervasyon_sorgula": rezervasyon_sorgula,
                "rezervasyon_iptal": rezervasyon_iptal,
                "guncel_rezervasyonlari_goster": guncel_rezervasyonlari_goster,
                "mail_at": mail_at
            }

            res = func_map[func_name](**args)
            mesaj_gecmisi.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": func_name,
                "content": json.dumps(res)
            })

        final_res = client.chat.completions.create(model="gpt-4o-mini", messages=mesaj_gecmisi)
        final_msg = final_res.choices[0].message
        mesaj_gecmisi.append(final_msg)
        return final_msg.content

    return response_message.content


Otel ajanınını oluşturdum

In [15]:
print(sohbet_et("10-15 mayıs arası bir rezervasyon oluşturmak isityorum."))

Ne kadar kişi olacağınızı ve hangi oda tipini (Single, Double, Suite) tercih ettiğinizi öğrenebilir miyim?


In [16]:
print(sohbet_et("2 kişi double istiyorum."))

10-15 Mayıs tarihleri arasında 2 kişi için Double oda seçeneğinizle birkaç alternatif sunabilirim:

1. **Oda No**: A-101
   - **Tip**: Double
   - **Manzara**: Havuz
   - **Fiyat**: 1100 TL/gece

2. **Oda No**: A-102
   - **Tip**: Double
   - **Manzara**: Deniz
   - **Fiyat**: 1500 TL/gece

3. **Oda No**: A-104
   - **Tip**: Double
   - **Manzara**: Deniz
   - **Fiyat**: 1250 TL/gece

Her 2 kişilik odada yangın alarmı ve klima bulunmaktadır. Hangi odayı tercih edersiniz?


In [17]:
print(sohbet_et("A-102 numaralı oda ugundur rezervasyon işlemini erçekleştirir misin?"))

A-102 numaralı Deniz manzaralı Double odanızı 10-15 Mayıs tarihleri arasında başarıyla rezerve ettim. Toplam maliyet 7500 TL (1500 TL/gece) olacaktır. 

Her zaman olduğu gibi odada yangın alarmı ve klima mevcuttur. 

İyi günler dilerim, daha fazla bilgi edinmek için müşteri hizmetlerimizle iletişime geçebilirsiniz.


In [18]:
print(sohbet_et("Bu rezervasyon bilgilerini ikbalktlk@gmail.com adresine mail olarak gönderir misin?"))

Rezervasyon bilgilerinizi ikbalktlk@gmail.com adresine başarılı bir şekilde gönderdim. Herhangi bir sorunuz veya başka bir isteğiniz olursa lütfen çekinmeden iletin. 

İyi günler dilerim!
